# Generate or load the 42-bin mean-field data

This notebook has two explicit modes:

- `MODE = "load"` checks the archived manuscript data and returns immediately.
- `MODE = "recompute"` starts from the 42-dimensional mean-field equations in `src/mean_field_42bin.py`.

For a first test, keep `QUICK_TEST = True`. The full manuscript grids are much more expensive.


In [ ]:
from pathlib import Path
import importlib
import os
import sys

os.environ['B_CUTOFF'] = '0.56'
os.environ['BIMODALITY_MODE'] = 'sample_corrected'
os.environ['STATE_MODE'] = 'three'
os.environ['SEPARATE_C_NEAR_LINES'] = '1'

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'src' / 'mean_field_42bin.py').is_file():
    raise RuntimeError('Run this notebook from the repository root or notebooks folder.')
sys.path.insert(0, str(root / 'src'))

import mean_field_42bin as mf
importlib.reload(mf)


## Choose the calculation mode

`TRACE_BOUNDARIES=True` runs the pseudo-arclength continuation after the stable-attractor scan.


In [ ]:
MODE = 'load'          # change to 'recompute' to calculate from scratch
QUICK_TEST = True      # use False only for the full grids
TRACE_BOUNDARIES = True

if MODE not in {'load', 'recompute'}:
    raise ValueError("MODE must be 'load' or 'recompute'.")


## Run

Fresh results are written to `data/generated_42bin/` and `figures/generated_42bin/`. The archived manuscript data are not modified.


In [ ]:
if MODE == 'load':
    mf.check_data()
    print('All archived 42-bin manuscript data are present:')
    for path in mf.required_data_files():
        print(f'  {path.relative_to(root)}  ({path.stat().st_size / 1024**2:.2f} MB)')
else:
    results = mf.recompute_all_42bin(
        output_dir=root / 'data' / 'generated_42bin',
        figure_dir=root / 'figures' / 'generated_42bin',
        quick=QUICK_TEST,
        trace_boundaries=TRACE_BOUNDARIES,
    )
    for panel in results:
        print()
        print('Generated:')
        for kind, path in panel.items():
            print(f'  {kind}: {path.relative_to(root)}')
